# 09n FastReID SBS R50-IBN Fine-Tune on CityFlowV2

Fine-tune the FastReID SBS ResNet50-IBN-a checkpoint on CityFlowV2 using the attached 09l training-data kernel output, then export the best checkpoint for downstream MTMC ensemble experiments.

## Recipe
1. Clone the project from `feature/pretrained-ensemble`
2. Download the VeRi SBS R50-IBN weights from the FastReID GitHub release
3. Remap FastReID checkpoint keys into `ReIDModelResNet50IBN` from `src/training/model.py`
4. Train with GeM + BNNeck, PK sampling (`16 IDs x 4 instances`), AdamW, cosine annealing, and mixed precision
5. Freeze the backbone for epochs 1-5, then unfreeze the full backbone while keeping GeM `p` frozen
6. Evaluate every 10 epochs with cosine-distance mAP / Rank-1 on the CityFlowV2 query/gallery split and save the best `state_dict`

## Outputs
- `/kaggle/working/fastreid_r50_ibn_cityflowv2_best.pth`
- `/kaggle/working/09n_fastreid_r50_finetune/training_history.json`
- `/kaggle/working/09n_fastreid_r50_finetune/final_metrics.json`

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip_install("loguru", "omegaconf", "scikit-learn", "matplotlib")

import torch

gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
runtime_summary = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
    "gpus": gpu_names,
}
print(json.dumps(runtime_summary, indent=2))

In [ ]:
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
REPO_URL = "https://github.com/MRKDaGods/gp.git"
BRANCH = "feature/pretrained-ensemble"

if not PROJECT.exists():
    print(f"Cloning {REPO_URL} [{BRANCH}] -> {PROJECT}")
    subprocess.check_call([
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        BRANCH,
        REPO_URL,
        str(PROJECT),
    ])
else:
    print(f"Project already present at {PROJECT}")

os.chdir(PROJECT)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

requirements_path = PROJECT / "requirements.txt"
if requirements_path.exists():
    pip_install("-r", str(requirements_path))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"])

try:
    import torchreid  # noqa: F401
except ImportError:
    pip_install("git+https://github.com/KaiyangZhou/deep-person-reid.git")

print(json.dumps({
    "cwd": os.getcwd(),
    "project": str(PROJECT),
    "branch": BRANCH,
    "requirements_found": requirements_path.exists(),
}, indent=2))

In [ ]:
import random
import urllib.request

import numpy as np

INPUT_ROOT = Path("/kaggle/input")

def looks_like_reid_root(path: Path) -> bool:
    return path.is_dir() and all((path / split).is_dir() for split in ["train", "query", "gallery"])

def discover_reid_root() -> tuple[Path, list[str]]:
    candidates = []
    if INPUT_ROOT.exists():
        for mount in sorted(INPUT_ROOT.iterdir()):
            if not mount.is_dir():
                continue
            probe_paths = [mount, mount / "cityflowv2_reid"]
            try:
                probe_paths.extend(child for child in mount.iterdir() if child.is_dir())
            except OSError:
                pass
            for probe in probe_paths:
                if looks_like_reid_root(probe):
                    candidates.append(probe)
    if not candidates and INPUT_ROOT.exists():
        for probe in INPUT_ROOT.rglob("*"):
            if looks_like_reid_root(probe):
                candidates.append(probe)
                if len(candidates) >= 10:
                    break
    if not candidates:
        raise FileNotFoundError("Could not find a mounted CityFlowV2 ReID split under /kaggle/input. Attach gumfreddy/mtmc-09l-cityflow-training-data.")
    unique = {str(candidate): candidate for candidate in candidates}
    ranked = sorted(unique.values(), key=lambda candidate: (0 if "09l" in str(candidate).lower() else 1, len(str(candidate))))
    return ranked[0], [str(candidate) for candidate in ranked[:5]]

REID_ROOT, REID_CANDIDATES = discover_reid_root()
OUTPUT_DIR = Path("/kaggle/working/09n_fastreid_r50_finetune")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
BEST_MODEL_PATH = Path("/kaggle/working/fastreid_r50_ibn_cityflowv2_best.pth")
LAST_CHECKPOINT_PATH = CHECKPOINT_DIR / "last_checkpoint.pth"
HISTORY_PATH = OUTPUT_DIR / "training_history.json"
METRICS_PATH = OUTPUT_DIR / "final_metrics.json"
for directory in [OUTPUT_DIR, CHECKPOINT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

IMG_SIZE = (256, 256)
BATCH_SIZE = 64
P_IDS = 16
K_INSTANCES = 4
NUM_WORKERS = 4
EPOCHS = 200
WARMUP_EPOCHS = 10
FREEZE_BACKBONE_EPOCHS = 5
EVAL_EVERY = 10

BACKBONE_LR = 3e-5
HEAD_LR = 5e-4
WEIGHT_DECAY = 5e-4
LABEL_SMOOTHING = 0.05
TRIPLET_MARGIN = 0.3
TRIPLET_WEIGHT = 0.3
CENTER_WEIGHT = 5e-4
CENTER_START_EPOCH = 15
GEM_P = 3.0
NUM_EXPECTED_CLASSES = 576
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE.startswith("cuda")

SBS_WEIGHTS_URL = "https://github.com/JDAI-CV/fast-reid/releases/download/v0.1.1/veri_sbs_R50-ibn.pth"
SBS_WEIGHTS_PATH = OUTPUT_DIR / "veri_sbs_R50-ibn.pth"
if not SBS_WEIGHTS_PATH.exists():
    print(f"Downloading FastReID SBS weights -> {SBS_WEIGHTS_PATH}")
    urllib.request.urlretrieve(SBS_WEIGHTS_URL, SBS_WEIGHTS_PATH)

print(json.dumps({
    "reid_root": str(REID_ROOT),
    "reid_candidates": REID_CANDIDATES,
    "weights_path": str(SBS_WEIGHTS_PATH),
    "img_size": IMG_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "device": DEVICE,
}, indent=2))

In [ ]:
from torch.utils.data import DataLoader

from src.training.datasets import PKSampler, ReIDDataset, build_test_transforms, build_train_transforms, parse_cityflowv2

train_data, query_data, gallery_data = parse_cityflowv2(str(REID_ROOT))
NUM_CLASSES = len({pid for _, pid, _ in train_data})
NUM_CAMERAS = len({cam for _, _, cam in (train_data + query_data + gallery_data)})
assert NUM_CLASSES == NUM_EXPECTED_CLASSES, f"Expected {NUM_EXPECTED_CLASSES} training IDs, found {NUM_CLASSES}"
assert query_data and gallery_data, "Query/gallery split is empty"

train_tf = build_train_transforms(
    height=IMG_SIZE[0],
    width=IMG_SIZE[1],
    random_erasing_prob=0.5,
    color_jitter=False,
)
test_tf = build_test_transforms(height=IMG_SIZE[0], width=IMG_SIZE[1])

train_ds = ReIDDataset(train_data, train_tf)
query_ds = ReIDDataset(query_data, test_tf)
gallery_ds = ReIDDataset(gallery_data, test_tf)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=PKSampler(train_data, p=P_IDS, k=K_INSTANCES),
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
query_loader = DataLoader(
    query_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    persistent_workers=NUM_WORKERS > 0,
)
gallery_loader = DataLoader(
    gallery_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=USE_AMP,
    persistent_workers=NUM_WORKERS > 0,
)

print(json.dumps({
    "train_images": len(train_data),
    "query_images": len(query_data),
    "gallery_images": len(gallery_data),
    "num_classes": NUM_CLASSES,
    "num_cameras": NUM_CAMERAS,
    "train_batches": len(train_loader),
    "query_batches": len(query_loader),
    "gallery_batches": len(gallery_loader),
}, indent=2))

In [ ]:
import torch.nn as nn

from src.training.model import ReIDModelResNet50IBN

def unwrap_checkpoint_state_dict(checkpoint):
    if isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            return checkpoint["state_dict"]
        if "model" in checkpoint:
            return checkpoint["model"]
        return checkpoint
    raise TypeError(f"Unsupported checkpoint type: {type(checkpoint)!r}")

def remap_fastreid_sbs_r50_ibn_state_dict(state_dict):
    remapped = {}
    skip_prefixes = ("pixel_mean", "pixel_std", "heads.classifier")
    for key, value in state_dict.items():
        key = key.replace("module.", "", 1)
        if key.startswith(skip_prefixes):
            continue
        if key.startswith("backbone.NL_"):
            continue
        if key == "heads.pool_layer.p":
            remapped["pool.p"] = value
            continue
        if key.startswith("heads.bottleneck."):
            suffix = key[len("heads.bottleneck."):]
            if suffix.startswith("0."):
                suffix = suffix[2:]
            remapped[f"bottleneck.{suffix}"] = value
            continue
        if key == "heads.bnneck.num_batches_tracked":
            remapped["bottleneck.num_batches_tracked"] = value
            continue
        if key.startswith("backbone."):
            remapped[key] = value
    return remapped

model = ReIDModelResNet50IBN(
    num_classes=NUM_CLASSES,
    last_stride=1,
    pretrained=False,
    gem_p=GEM_P,
    eval_feature="global",
)
checkpoint = torch.load(SBS_WEIGHTS_PATH, map_location="cpu", weights_only=False)
state_dict = remap_fastreid_sbs_r50_ibn_state_dict(unwrap_checkpoint_state_dict(checkpoint))
missing, unexpected = model.load_state_dict(state_dict, strict=False)
nn.init.normal_(model.classifier.weight, std=0.001)
model.pool.p.requires_grad_(False)
model = model.to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
base_model = model.module if hasattr(model, "module") else model
critical_missing = [key for key in missing if not key.startswith("classifier")]

print(json.dumps({
    "loaded_tensors": len(state_dict),
    "critical_missing": critical_missing[:20],
    "unexpected": unexpected[:20],
    "pool_p": float(base_model.pool.p.detach().cpu().item()),
    "pool_p_frozen": not base_model.pool.p.requires_grad,
    "feat_dim": base_model.feat_dim,
    "eval_feature": base_model.eval_feature,
}, indent=2))

In [ ]:
from src.training.evaluate_reid import evaluate_reid
from src.training.losses import CenterLoss, CrossEntropyLabelSmooth, TripletLoss
from src.training.train_reid import build_cosine_scheduler

id_loss_fn = CrossEntropyLabelSmooth(NUM_CLASSES, epsilon=LABEL_SMOOTHING).to(DEVICE)
triplet_loss_fn = TripletLoss(margin=TRIPLET_MARGIN, soft_margin=True).to(DEVICE)
center_loss_fn = CenterLoss(NUM_CLASSES, 2048).to(DEVICE)

def unwrap_model(current_model):
    return current_model.module if hasattr(current_model, "module") else current_model

def set_backbone_trainable(current_model, trainable: bool):
    raw_model = unwrap_model(current_model)
    for param in raw_model.backbone.parameters():
        param.requires_grad_(trainable)
    raw_model.pool.p.requires_grad_(False)

def build_optimizers(current_model):
    raw_model = unwrap_model(current_model)
    optimizer = torch.optim.AdamW(
        [
            {"params": raw_model.backbone.parameters(), "lr": BACKBONE_LR},
            {"params": raw_model.bottleneck.parameters(), "lr": HEAD_LR},
            {"params": raw_model.classifier.parameters(), "lr": HEAD_LR},
        ],
        betas=(0.9, 0.999),
        weight_decay=WEIGHT_DECAY,
    )
    center_optimizer = torch.optim.SGD(center_loss_fn.parameters(), lr=0.5)
    scheduler = build_cosine_scheduler(
        optimizer,
        warmup_epochs=WARMUP_EPOCHS,
        total_epochs=EPOCHS,
        eta_min=1e-7,
        start_factor=0.1,
    )
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    return optimizer, center_optimizer, scheduler, scaler

def evaluate_model(current_model):
    mAP, cmc, _, _ = evaluate_reid(
        current_model,
        query_loader,
        gallery_loader,
        device=DEVICE,
        rerank=False,
    )
    return {
        "mAP": float(mAP),
        "rank1": float(cmc[0]) if len(cmc) > 0 else 0.0,
        "rank5": float(cmc[4]) if len(cmc) > 4 else 0.0,
        "rank10": float(cmc[9]) if len(cmc) > 9 else 0.0,
    }

set_backbone_trainable(model, False)
optimizer, center_optimizer, scheduler, scaler = build_optimizers(model)
baseline_metrics = evaluate_model(model)

print(json.dumps({
    "baseline_metrics": baseline_metrics,
    "triplet_soft_margin": True,
    "triplet_weight": TRIPLET_WEIGHT,
    "center_weight": CENTER_WEIGHT,
    "center_start_epoch": CENTER_START_EPOCH,
    "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
    "optimizer_lrs": [group["lr"] for group in optimizer.param_groups],
}, indent=2))

In [ ]:
def save_training_state(epoch, best_mAP, best_epoch, history_records):
    torch.save({
        "epoch": epoch,
        "model": unwrap_model(model).state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_mAP": best_mAP,
        "best_epoch": best_epoch,
        "history": history_records,
    }, LAST_CHECKPOINT_PATH)

def train_one_epoch(epoch):
    raw_model = unwrap_model(model)
    if epoch == FREEZE_BACKBONE_EPOCHS + 1:
        set_backbone_trainable(model, True)
        print(f"Unfroze backbone at epoch {epoch}")

    model.train()
    use_center = epoch >= CENTER_START_EPOCH
    running = {"loss": 0.0, "id_loss": 0.0, "tri_loss": 0.0, "cen_loss": 0.0}

    for batch_idx, (imgs, pids, _, _) in enumerate(train_loader, start=1):
        imgs = imgs.to(DEVICE, non_blocking=USE_AMP)
        pids = pids.to(DEVICE).long()

        optimizer.zero_grad(set_to_none=True)
        if use_center:
            center_optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=USE_AMP):
            cls_score, global_feat, bn_feat = model(imgs)
            loss_id = id_loss_fn(cls_score, pids)
            loss_tri = triplet_loss_fn(global_feat, pids)
            loss_cen_raw = center_loss_fn(global_feat.float(), pids) if use_center else torch.tensor(0.0, device=global_feat.device)
            loss = loss_id + (TRIPLET_WEIGHT * loss_tri) + (CENTER_WEIGHT * loss_cen_raw)

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            if use_center:
                scaler.unscale_(center_optimizer)
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            if use_center:
                for param in center_loss_fn.parameters():
                    if param.grad is not None:
                        param.grad.data *= 1.0 / max(CENTER_WEIGHT, 1e-12)
            scaler.step(optimizer)
            if use_center:
                scaler.step(center_optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(raw_model.parameters(), max_norm=5.0)
            if use_center:
                for param in center_loss_fn.parameters():
                    if param.grad is not None:
                        param.grad.data *= 1.0 / max(CENTER_WEIGHT, 1e-12)
            optimizer.step()
            if use_center:
                center_optimizer.step()

        running["loss"] += float(loss.detach().item())
        running["id_loss"] += float(loss_id.detach().item())
        running["tri_loss"] += float(loss_tri.detach().item())
        running["cen_loss"] += float(loss_cen_raw.detach().item()) if use_center else 0.0

        if batch_idx % 10 == 0 or batch_idx == len(train_loader):
            avg_loss = running["loss"] / batch_idx
            avg_id = running["id_loss"] / batch_idx
            avg_tri = running["tri_loss"] / batch_idx
            avg_cen = running["cen_loss"] / batch_idx
            print(
                f"Epoch {epoch:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | "
                f"loss={avg_loss:.4f} id={avg_id:.4f} tri={avg_tri:.4f} cen={avg_cen:.4f} "
                f"lr_backbone={optimizer.param_groups[0]['lr']:.2e} lr_head={optimizer.param_groups[1]['lr']:.2e}"
            )

    scheduler.step()
    num_batches = max(len(train_loader), 1)
    return {key: value / num_batches for key, value in running.items()}

In [ ]:
history = []
best_mAP = -1.0
best_epoch = 0
best_metrics = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = train_one_epoch(epoch)
    record = {
        "epoch": epoch,
        "lr_backbone": float(optimizer.param_groups[0]["lr"]),
        "lr_head": float(optimizer.param_groups[1]["lr"]),
        **{key: float(value) for key, value in train_metrics.items()},
    }

    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
        eval_metrics = evaluate_model(model)
        record.update(eval_metrics)
        print(
            f"Eval epoch {epoch:03d} | mAP={eval_metrics['mAP']:.4f} "
            f"R1={eval_metrics['rank1']:.4f} R5={eval_metrics['rank5']:.4f} R10={eval_metrics['rank10']:.4f}"
        )
        if eval_metrics["mAP"] > best_mAP:
            best_mAP = eval_metrics["mAP"]
            best_epoch = epoch
            best_metrics = dict(eval_metrics)
            torch.save(unwrap_model(model).state_dict(), BEST_MODEL_PATH)
            print(f"Saved new best checkpoint -> {BEST_MODEL_PATH}")

    history.append(record)
    with HISTORY_PATH.open("w", encoding="utf-8") as handle:
        json.dump(history, handle, indent=2)
    save_training_state(epoch, best_mAP, best_epoch, history)

assert BEST_MODEL_PATH.exists(), f"Missing best checkpoint: {BEST_MODEL_PATH}"
print(json.dumps({
    "best_epoch": best_epoch,
    "best_mAP": best_mAP,
    "history_path": str(HISTORY_PATH),
    "last_checkpoint_path": str(LAST_CHECKPOINT_PATH),
}, indent=2))

In [ ]:
best_state = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
unwrap_model(model).load_state_dict(best_state, strict=True)
final_eval_metrics = evaluate_model(model)

final_payload = {
    "reid_root": str(REID_ROOT),
    "weights_path": str(SBS_WEIGHTS_PATH),
    "best_model_path": str(BEST_MODEL_PATH),
    "best_epoch": best_epoch,
    "best_metrics": best_metrics if best_metrics is not None else final_eval_metrics,
    "final_eval_metrics": final_eval_metrics,
    "baseline_metrics": baseline_metrics,
    "config": {
        "img_size": IMG_SIZE,
        "batch_size": BATCH_SIZE,
        "p_ids": P_IDS,
        "k_instances": K_INSTANCES,
        "epochs": EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "freeze_backbone_epochs": FREEZE_BACKBONE_EPOCHS,
        "backbone_lr": BACKBONE_LR,
        "head_lr": HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "label_smoothing": LABEL_SMOOTHING,
        "triplet_soft_margin": True,
        "triplet_weight": TRIPLET_WEIGHT,
        "center_weight": CENTER_WEIGHT,
        "center_start_epoch": CENTER_START_EPOCH,
    },
}
with METRICS_PATH.open("w", encoding="utf-8") as handle:
    json.dump(final_payload, handle, indent=2)

print(json.dumps(final_payload, indent=2))

In [ ]:
state_dict = torch.load(BEST_MODEL_PATH, map_location="cpu", weights_only=False)
required_prefixes = [
    "backbone.conv1",
    "backbone.layer1",
    "backbone.layer2",
    "backbone.layer3",
    "backbone.layer4",
    "pool.p",
    "bottleneck",
    "classifier",
]
prefix_hits = {prefix: any(key.startswith(prefix) for key in state_dict) for prefix in required_prefixes}
assert all(prefix_hits.values()), prefix_hits

unwrap_model(model).eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE[0], IMG_SIZE[1], device=DEVICE)
    embeddings = unwrap_model(model)(dummy)
assert tuple(embeddings.shape) == (2, 2048), tuple(embeddings.shape)

print(json.dumps({
    "best_model_path": str(BEST_MODEL_PATH),
    "parameter_tensors": len(state_dict),
    "prefix_hits": prefix_hits,
    "eval_embedding_shape": list(embeddings.shape),
    "history_path": str(HISTORY_PATH),
    "metrics_path": str(METRICS_PATH),
}, indent=2))